# M0.2 · Matris çarpımı, shape ve broadcasting

Her sinir ağı katmanının çekirdeği olan işlemi 2×2'de elle, sonra üç iç içe döngüyle, NumPy'da, PyTorch'ta autograd ile ve en sonda yüzlerce matrisi tek çağrıda çarparak kuruyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m0-2-matmul.html](https://m-gonul.github.io/pytorch-lab/m0-2-matmul.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m0_2_matmul.py
CHECKS = {
 "b": "import random\nfrom labcheck import close, no_imports, need\n\n\ndef ref(A, B):\n    return [[sum(A[i][t] * B[t][j] for t in range(len(B))) for j in range(len(B[0]))] for i in range(len(A))]\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\", \"pandas\"}))\n    R.test(\"`matmul` tanımlı\", lambda: need(ns, \"matmul\"))\n    matmul = ns.get(\"matmul\")\n    if matmul is None:\n        return\n    R.test(\"elle örnek: [[19,22],[43,50]]\", lambda: close(matmul([[1, 2], [3, 4]], [[5, 6], [7, 8]]), [[19, 22], [43, 50]]))\n    R.test(\"1×1\", lambda: close(matmul([[3]], [[-2]]), [[-6]]))\n\n    def rect():\n        rng = random.Random(0)\n        for n, k, m in [(3, 4, 2), (1, 5, 3), (4, 1, 4)]:\n            A = [[rng.uniform(-2, 2) for _ in range(k)] for _ in range(n)]\n            B = [[rng.uniform(-2, 2) for _ in range(m)] for _ in range(k)]\n            close(matmul(A, B), ref(A, B), what=f\"({n}×{k})({k}×{m})\")\n    R.test(\"dikdörtgen matrisler (3×4·4×2, 1×5·5×3, 4×1·1×4)\", rect)\n\n    def mismatch():\n        try:\n            matmul([[1, 2, 3]], [[1, 2], [3, 4]])\n        except ValueError:\n            return\n        raise AssertionError(\"(1×3)(2×2) için ValueError bekleniyordu\")\n    R.test(\"shape uyumsuzluğunda ValueError\", mismatch)\n\n    def pure():\n        A, B = [[1, 2], [3, 4]], [[5, 6], [7, 8]]\n        matmul(A, B)\n        assert A == [[1, 2], [3, 4]] and B == [[5, 6], [7, 8]], \"girdiler değiştirilmiş\"\n    R.test(\"girdiler değişmiyor\", pure)\n",
 "c": "import numpy as np\nfrom labcheck import close, no_loops, no_imports, need\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü/comprehension yok\", lambda: no_loops(src))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    rng = np.random.default_rng(0)\n\n    def mm():\n        f = need(ns, \"matmul_np\")\n        close(f(np.array([[1, 2], [3, 4]]), np.array([[5, 6], [7, 8]])), [[19, 22], [43, 50]])\n        A, B = rng.normal(size=(7, 5)), rng.normal(size=(5, 3))\n        out = f(A, B)\n        assert isinstance(out, np.ndarray), \"np.ndarray döndürmeli\"\n        close(out, A @ B, what=\"(7×5)(5×3)\")\n    R.test(\"matmul_np doğru\", mm)\n\n    def cc():\n        f = need(ns, \"center_columns\")\n        X = rng.normal(loc=3.0, size=(6, 4))\n        out = f(X.copy())\n        close(out, X - X.mean(0), what=\"center_columns\")\n        close(np.asarray(out).mean(0), np.zeros(4), atol=1e-9, what=\"sütun ortalamaları\")\n    R.test(\"center_columns: sütun ortalamaları 0\", cc)\n",
 "d": "import torch\nfrom labcheck import close, need, uses_any, ensure\n\n\ndef check(ns, R):\n    g = torch.Generator().manual_seed(0)\n    A = torch.randn(4, 3, generator=g, dtype=torch.float64)\n    B = torch.randn(3, 5, generator=g, dtype=torch.float64)\n\n    def mm():\n        out = need(ns, \"matmul_torch\")(A, B)\n        assert torch.is_tensor(out), \"torch.Tensor döndürmeli\"\n        close(out, A @ B, what=\"(4×3)(3×5)\")\n    R.test(\"matmul_torch doğru\", mm)\n\n    def grads():\n        res = need(ns, \"matmul_grads\")(A, B)\n        assert isinstance(res, (tuple, list)) and len(res) == 2, \"(dA, dB) çifti döndürmeli\"\n        dA, dB = res\n        G = torch.ones(4, 5, dtype=torch.float64)\n        close(dA, G @ B.T, what=\"dL/dA\")\n        close(dB, A.T @ G, what=\"dL/dB\")\n        return \"dL/dA = 1·Bᵀ ve dL/dB = Aᵀ·1 ile eşleşti\"\n    R.test(\"gradyanlar formülle eşleşiyor\", grads)\n\n    R.test(\"autograd kullanılıyor (.backward / torch.autograd.grad)\",\n           lambda: ensure(uses_any(ns[\"__source__\"], [\"backward\", \"grad\"]), \"gradyanı elle yazma; L.backward() kullan\"))\n",
 "e": "import random\nimport numpy as np\nimport torch\nfrom labcheck import close, no_loops, need, ensure, bench, fmt_time\n\n\ndef py_matmul(A, B):\n    n, k, m = len(A), len(B), len(B[0])\n    C = [[0.0] * m for _ in range(n)]\n    for i in range(n):\n        Ai, Ci = A[i], C[i]\n        for t in range(k):\n            a, Bt = Ai[t], B[t]\n            for j in range(m):\n                Ci[j] += a * Bt[j]\n    return C\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    R.test(\"`batched_matmul` tanımlı\", lambda: need(ns, \"batched_matmul\"))\n    bmm = ns.get(\"batched_matmul\")\n    if bmm is None:\n        return\n    g = torch.Generator().manual_seed(0)\n    As = torch.randn(32, 4, 6, generator=g)\n    Bs = torch.randn(32, 6, 3, generator=g)\n    R.test(\"(32,4,6)·(32,6,3) doğru\", lambda: close(bmm(As, Bs), torch.stack([a @ b for a, b in zip(As, Bs)]), rtol=1e-4, atol=1e-5))\n    R.test(\"çıktı shape'i (N, n, m)\", lambda: ensure(tuple(bmm(As, Bs).shape) == (32, 4, 3), f\"shape {tuple(bmm(As, Bs).shape)}\"))\n\n    # --- hız tablosu 1: tek 64×64 çarpım\n    n = 64\n    rng = random.Random(0)\n    A = [[rng.random() for _ in range(n)] for _ in range(n)]\n    B = [[rng.random() for _ in range(n)] for _ in range(n)]\n    An, Bn = np.array(A), np.array(B)\n    At, Bt = torch.tensor(A), torch.tensor(B)\n    t_py = bench(lambda: py_matmul(A, B), repeat=3, number=1)\n    t_np = bench(lambda: An @ Bn)\n    t_th = bench(lambda: At @ Bt)\n    R.table(f\"Tek çarpım, {n}×{n} ({n**3:,} çarpma)\", [\"yöntem\", \"süre\", \"saf Python'a göre\"],\n            [[\"saf Python döngüsü\", fmt_time(t_py), \"1×\"],\n             [\"NumPy  A @ B\", fmt_time(t_np), f\"{t_py / t_np:,.0f}× hızlı\"],\n             [\"PyTorch  A @ B\", fmt_time(t_th), f\"{t_py / t_th:,.0f}× hızlı\"]])\n\n    # --- hız tablosu 2: 256 küçük çarpım, döngü vs batched\n    As2, Bs2 = torch.randn(256, 8, 8), torch.randn(256, 8, 8)\n    t_loop = bench(lambda: torch.stack([a @ b for a, b in zip(As2, Bs2)]))\n    t_b = bench(lambda: bmm(As2, Bs2))\n    t_at = bench(lambda: As2 @ Bs2)\n    R.table(\"256 adet 8×8 çarpım\", [\"yöntem\", \"süre\", \"döngüye göre\"],\n            [[\"for + @ (256 kernel çağrısı)\", fmt_time(t_loop), \"1×\"],\n             [\"senin batched_matmul\", fmt_time(t_b), f\"{t_loop / t_b:,.1f}× hızlı\"],\n             [\"As @ Bs\", fmt_time(t_at), f\"{t_loop / t_at:,.1f}× hızlı\"]],\n            note=\"Küçük matrislerde maliyetin çoğu hesap değil, çağrı başına sabit maliyet (yorumlayıcı + dispatch).\")\n"
}

In [ ]:
import importlib, labcheck, checks_m0_2_matmul as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Bir sinir ağında zamanın büyük kısmı tek bir işlemde geçer: matris çarpımı. Bir lineer katman $y = xW^\top + b$, bir attention skoru $QK^\top$, bir convolution (im2col sonrası) — hepsi aynı çekirdeğe iner.

$$
C = AB,\qquad C_{ij} = \sum_{t=1}^{k} A_{it}\,B_{tj},\qquad (n\times k)(k\times m) \to (n\times m)
$$

Her $C_{ij}$, $A$'nın $i$. satırı ile $B$'nin $j$. sütununun iç çarpımıdır (M0.1). Toplam iş $n\cdot k\cdot m$ çarpma ve yaklaşık o kadar toplamadır; $n=k=m$ için $O(n^3)$.

İkinci kavram broadcasting: shape'leri farklı iki dizi arasında eleman eleman işlem yaparken, küçük olanın eksik boyutlarının "kopyalanmış gibi" davranması. Kural: boyutlar **sağdan** hizalanır; her eksende ya eşittirler ya biri 1'dir (ya da yoktur). Kopya bellekte yapılmaz.

## 1 · Elle

Kâğıt ve kalemle çalış; hesap makinesi yok. Her hücreyi gir, "Kontrol et" ile tek tek doğrula.

<!-- çözüm -->

$$
A=\begin{bmatrix}1&2\\3&4\end{bmatrix},\qquad
B=\begin{bmatrix}5&6\\7&8\end{bmatrix}
$$

Önce $C_{11}$: $A$'nın 1. satırı $[1,2]$ ile $B$'nin 1. sütunu $[5,7]$'nin iç çarpımı. Kaç çarpma yaptığını da say; bu sayı (e) aşamasında ölçeceğin süreyle doğrudan bağlantılı.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**$C = AB$ hücre hücre**

- `c11`: $C_{11}$
- `c12`: $C_{12}$
- `c21`: $C_{21}$
- `c22`: $C_{22}$

**Maliyet ve shape**

- `mults`: $C=AB$ için toplam skaler çarpma sayısı
- `mults_big`: $(64\times 64)(64\times 64)$ için skaler çarpma sayısı
- `rows`: $(2\times 3)(3\times 4)$ sonucunun satır sayısı
- `cols`: … sütun sayısı

**Broadcasting: $A + v$, $v=[10,\;20]$**

- `b11`: $(A+v)_{11}$
- `b12`: $(A+v)_{12}$
- `b21`: $(A+v)_{21}$
- `b22`: $(A+v)_{22}$

In [ ]:
cevaplar = dict(
    c11=None,
    c12=None,
    c21=None,
    c22=None,
    mults=None,
    mults_big=None,
    rows=None,
    cols=None,
    b11=None,
    b12=None,
    b21=None,
    b22=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"c11":[19,0],"c12":[22,0],"c21":[43,0],"c22":[50,0],"mults":[8,0],"mults_big":[262144,0],"rows":[2,0],"cols":[4,0],"b11":[11,0],"b12":[22,0],"b21":[13,0],"b22":[24,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Şimdi aynı tanımı Python listeleriyle yaz. Matris, satırlardan oluşan bir liste: `[[1, 2], [3, 4]]`.

- `matmul(A, B)` iki matrisi çarpsın ve yeni bir liste döndürsün; girdileri değiştirmesin.
- İç boyutlar uyuşmazsa `ValueError` fırlatsın.
- `numpy` ve `torch` yasak. Üç iç içe döngü: $i$ (satır), $j$ (sütun), $t$ (toplam indisi).

Döngü sırasına dikkat: `for i … for j … for t` en okunaklı olanı; `i, t, j` sırası $B$'yi satır satır gezdiği için daha "önbellek dostu"dur. Burada fark etmez ama (e)'de C seviyesinde bunun büyük etkisi var.

In [ ]:
%%writefile stage_b.py
def matmul(A, B):
    """A: n×k, B: k×m (iç içe listeler). n×m liste döndür."""
    # 1) shape'leri oku, iç boyut uyuşmazsa ValueError
    # 2) sıfırlarla dolu n×m sonuç oluştur
    # 3) üç iç içe döngü ile C[i][j] += A[i][t] * B[t][j]
    raise NotImplementedError


print(matmul([[1, 2], [3, 4]], [[5, 6], [7, 8]]))  # [[19, 22], [43, 50]]

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy'da $C=AB$ tek operatördür: `A @ B` (ya da `np.matmul`). Bu aşamada **döngü ve comprehension yasak**; kontrol bunu kaynak kodun AST'sinden (soyut sözdizimi ağacı) denetler.

İki fonksiyon yaz:

- `matmul_np(A, B)` — `np.ndarray` döndürsün.
- `center_columns(X)` — her sütundan o sütunun ortalamasını çıkarsın. Broadcasting ile tek satır: `X.mean(axis=0)` shape'i `(k,)` ve `(n, k)` ile sağdan hizalanır.

pandas'ın burada yeri küçük: aynı işlem `df - df.mean()`; pandas sütunları **isimleriyle** hizalar, NumPy **pozisyonla**. Kodun altına bunu deneyen bir satır eklemekte serbestsin.

In [ ]:
%%writefile stage_c.py
import numpy as np


def matmul_np(A, B):
    """Döngüsüz matris çarpımı."""
    ...


def center_columns(X):
    """Her sütundan kendi ortalamasını çıkar (broadcasting)."""
    ...


A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
print(matmul_np(A, B))
print(center_columns(np.array([[1.0, 10.0], [3.0, 30.0]])))  # [[-1,-10],[1,10]]

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

PyTorch'ta `@` aynı işi yapar; fark, autograd'ın işlemi kaydedip gradyanı hesaplayabilmesi.

$L = \sum_{ij} C_{ij}$, $C = AB$ olsun. Zincir kuralıyla, $G = \partial L/\partial C$ (burada her elemanı 1 olan $n\times m$ matris) için

$$
\frac{\partial L}{\partial A} = G\,B^\top \quad (n\times k),\qquad
\frac{\partial L}{\partial B} = A^\top G \quad (k\times m).
$$

Bu iki formül, her lineer katmanın backward'ının ta kendisi. Yaz:

- `matmul_torch(A, B)` — iki tensörü çarpsın.
- `matmul_grads(A, B)` — `A` ve `B`'nin kopyalarını `requires_grad=True` yapıp $L$'yi hesaplasın, `L.backward()` çağırsın ve `(dA, dB)` döndürsün.

Kontrol, döndürdüğün gradyanları yukarıdaki formülle karşılaştırır.

In [ ]:
%%writefile stage_d.py
import torch


def matmul_torch(A, B):
    ...


def matmul_grads(A, B):
    """L = (A @ B).sum() için (dL/dA, dL/dB) döndür — autograd ile."""
    A = A.detach().clone().requires_grad_(True)
    B = B.detach().clone().requires_grad_(True)
    # L'yi hesapla, backward çağır, gradyanları döndür
    ...


A = torch.tensor([[1., 2.], [3., 4.]])
B = torch.tensor([[5., 6.], [7., 8.]])
print(matmul_torch(A, B))
print(matmul_grads(A, B))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Gerçek iş yükünde tek bir matris değil, bir **batch** (yığın) matris çarpılır: $N$ adet $(n\times k)$ ile $N$ adet $(k\times m)$. Python'da `for` ile $N$ kez `@` çağırmak her adımda yorumlayıcı ve kernel başlatma maliyeti öder.

`batched_matmul(As, Bs)` yaz: `As.shape == (N, n, k)`, `Bs.shape == (N, k, m)`, sonuç `(N, n, m)`. **Döngü yasak.** Üç eşdeğer yol:

```python
As @ Bs                              # matmul baştaki boyutları batch sayar
torch.bmm(As, Bs)                    # yalnız 3-boyutlu, açık batch
torch.einsum("bik,bkj->bij", As, Bs)  # indis notasyonu: tanımın kendisi
```

`einsum` dizgesi (b) aşamasındaki döngünün harfiyen tarifidir: tekrar eden ama çıktıda olmayan `k` toplanır.

Kontrol ayrıca bir **hız tablosu** üretir: aynı $64\times64$ çarpımı saf Python döngüsü, NumPy ve PyTorch ile; ve 256 küçük matris için döngü vs batched. Sonuçları yorumla: hızlanmanın ne kadarı yorumlayıcı maliyetinin kalkmasından, ne kadarı BLAS çekirdeklerinden geliyor?

In [ ]:
%%writefile stage_e.py
import torch


def batched_matmul(As, Bs):
    """As: (N, n, k), Bs: (N, k, m) → (N, n, m). Döngü yok."""
    ...


As = torch.randn(256, 8, 8)
Bs = torch.randn(256, 8, 8)
print(batched_matmul(As, Bs).shape)  # torch.Size([256, 8, 8])

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Makale ödevi: Strassen (1969): Gauss eliminasyonu optimal değil

[Strassen — Gaussian Elimination is not Optimal (Numerische Mathematik 1969)](https://link.springer.com/article/10.1007/BF02165411)

Strassen'in 7 çarpımlı 2 × 2 blok formülünü özyinelemeli yaz (NumPy blokları); bir eşik altında np.matmul'a geç. n = 2⁶…2¹¹ için süreyi ve çarpma sayısını (n^2.807 vs n³) ölç, en iyi eşiği bul; float32/float64'te sonucun hata büyümesini klasik çarpımla karşılaştır (Strassen'in daha az kararlı olduğunu göster).


**Adımlar**

1. İki kuvveti olmayan boyutları sıfır dolgu ile ele al.

**Kabul ölçütleri**

- Eşik üstünde Strassen'in daha hızlı olduğu bir n (ya da olmadığına dair ölçüm).
- Hata tablosu: Strassen hatası n ile daha hızlı büyür.

## Kaynaklar

- [MIT 18.06 Linear Algebra (Strang) — Ders 3: Multiplication and inverse matrices](https://ocw.mit.edu/courses/18-06-linear-algebra-spring-2010/) — CC BY-NC-SA 4.0
- [3Blue1Brown — Matrix multiplication as composition](https://www.3blue1brown.com/lessons/matrix-multiplication) — ©
- [Dive into Deep Learning §2.3 Linear Algebra](https://d2l.ai/chapter_preliminaries/linear-algebra.html) — CC BY-SA 4.0 / MIT-0
- [CS231n Python/NumPy tutorial (broadcasting)](https://cs231n.github.io/python-numpy-tutorial/) — MIT
- [NumPy — Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html) — BSD-3
- [PyTorch — torch.einsum / torch.bmm](https://docs.pytorch.org/docs/stable/generated/torch.einsum.html) — BSD-3